# EXEMPLO 1: Hybrid Query Básico — BGE-M3 (Ollama)

**Objetivo**: Referência rápida para implementar busca híbrida (BM25 + Vector) no OpenSearch usando **BGE-M3 via Ollama** (dim=1024) — mesmo padrão da Aula 1.

Este notebook demonstra:
- Setup mínimo de conexão OpenSearch + Ollama
- Criação de índice híbrido **com dimensão 1024** (BGE-M3)
- Geração de embeddings client-side via Ollama (`bge-m3`)
- **Pipeline RRF com sintaxe oficial OpenSearch 2.19+** (`score-ranker-processor`)
- Query híbrida completa
- Snippet copiável para reutilizar

**Referência (RRF):** <https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/>

> **Pré-requisitos**: Ollama rodando em `http://localhost:11434` com o modelo `bge-m3` carregado (`ollama pull bge-m3`).

## Setup Mínimo

In [22]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opensearch-py', 'requests'])

import os, requests
from opensearchpy import OpenSearch
from typing import List, Dict, Any

client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False,
    verify_certs=False
)

OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL',  'http://localhost:11434')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL1', 'bge-m3')
print(f"OLLAMA_BASE_URL    = {OLLAMA_BASE_URL}")
print(f"OLLAMA_EMBED_MODEL = {OLLAMA_EMBED_MODEL}")
EMBED_DIM = 1024  # BGE-M3

print(f"OpenSearch: {client.info()['version']['number']}")
print(f"Ollama    : {OLLAMA_BASE_URL} ({OLLAMA_EMBED_MODEL}, dim={EMBED_DIM})")

OLLAMA_BASE_URL    = http://localhost:11434
OLLAMA_EMBED_MODEL = bge-m3
OpenSearch: 3.0.0
Ollama    : http://localhost:11434 (bge-m3, dim=1024)


## Função de Embedding (Ollama BGE-M3)

In [23]:
def ollama_embed(text: str, model: str = OLLAMA_EMBED_MODEL,
                 base_url: str = OLLAMA_BASE_URL) -> List[float]:
    """Gera embedding BGE-M3 via Ollama. Fallback: vetor zerado."""
    try:
        resp = requests.post(
            f"{base_url}/api/embeddings",
            json={"model": model, "prompt": text},
            timeout=30,
        )
        resp.raise_for_status()
        emb = resp.json().get("embedding", [])
        if len(emb) != EMBED_DIM:
            raise ValueError(f"Dim inesperada: {len(emb)} (esperado {EMBED_DIM})")
        return emb
    except Exception as e:
        print(f"⚠️  Ollama indisponível ({e}). Usando vetor zero.")
        return [0.0] * EMBED_DIM

vec = ollama_embed("teste de embedding")
print(f"✓ Dimensão do embedding: {len(vec)}")

✓ Dimensão do embedding: 1024


## Criar Índice Híbrido (dim=1024 para BGE-M3)

In [24]:
index_name = "indice_hibrido_exemplo1_bge_m3"

try:
    client.indices.delete(index=index_name)
except Exception:
    pass

mapping = {
    "settings": {"number_of_shards": 1, "index": {"knn": True}},
    "mappings": {
        "properties": {
            "texto": {"type": "text", "analyzer": "portuguese"},
            "embedding": {
                "type": "knn_vector",
                "dimension": EMBED_DIM,
                "method": {"name": "hnsw", "space_type": "cosinesimil", "engine": "faiss"}
            }
        }
    }
}

client.indices.create(index=index_name, body=mapping)
print(f"✓ Índice criado: {index_name} (dim={EMBED_DIM})")

✓ Índice criado: indice_hibrido_exemplo1_bge_m3 (dim=1024)


## Inserir Documentos

In [25]:
docs = [
    {"id": "1", "texto": "Lei de Acesso à Informação garante direito a informações públicas"},
    {"id": "2", "texto": "Direitos fundamentais incluem vida, liberdade, igualdade e segurança"},
    {"id": "3", "texto": "LGPD protege dados pessoais e privacidade dos cidadãos"},
]

for doc in docs:
    doc["embedding"] = ollama_embed(doc["texto"])
    client.index(index=index_name, id=doc["id"],
                 body={"texto": doc["texto"], "embedding": doc["embedding"]})

client.indices.refresh(index=index_name)
print(f"✓ {len(docs)} documentos indexados com embeddings BGE-M3")

✓ 3 documentos indexados com embeddings BGE-M3


## Criar Pipeline RRF (`score-ranker-processor`)

Sintaxe oficial OpenSearch 2.19+ — ver <https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/>

In [26]:
try:
    client.transport.perform_request('DELETE', '/_search/pipeline/rrf_pipeline')
except Exception:
    pass

pipeline_body = {
    "description": "Post processor for hybrid RRF search (BGE-M3 + BM25)",
    "phase_results_processors": [
        {
            "score-ranker-processor": {
                "combination": {
                    "technique": "rrf",
                    "rank_constant": 60   # k da fórmula 1/(k + rank); default 60
                }
            }
        }
    ]
}

client.transport.perform_request('PUT', '/_search/pipeline/rrf_pipeline', body=pipeline_body)
print("✓ Pipeline RRF criado (score-ranker-processor, rank_constant=60)")

✓ Pipeline RRF criado (score-ranker-processor, rank_constant=60)


## Query Híbrida Completa

In [30]:
def hybrid_query(client, index: str, query_text: str, size: int = 5) -> List[Dict]:
    """Busca híbrida: BM25 (texto) + KNN (BGE-M3) + RRF (fusão)."""
    query_embedding = ollama_embed(query_text)

    search_body = {
        "size": size,
        "query": {
            "hybrid": {
                "queries": [
                    {"match": {"texto": {"query": query_text}}},
                    {"knn": {"embedding": {"vector": query_embedding, "k": size}}}
                ]
            }
        }
    }

    print(search_body)

    response = client.search(
        index=index,
        body=search_body,
        params={"search_pipeline": "rrf_pipeline"},
    )

    return [
        {'id': hit['_id'], 'score': hit['_score'], 'texto': hit['_source']['texto']}
        for hit in response['hits']['hits']
    ]

query = "informação pública e privacidade"
resultados = hybrid_query(client, index_name, query, size=3)

print(f"\nQuery: '{query}'")
print(f"\nResultados ({len(resultados)}):")
for r in resultados:
    print(f"\n[{r['id']}] Score: {r['score']:.4f}")
    print(f"    {r['texto']}")

{'size': 3, 'query': {'hybrid': {'queries': [{'match': {'texto': {'query': 'informação pública e privacidade'}}}, {'knn': {'embedding': {'vector': [-1.4109448194503784, 0.6242245435714722, -0.9415174722671509, 0.24010182917118073, -0.4805283546447754, -0.20891430974006653, 0.7005484104156494, -0.14598944783210754, 1.0444892644882202, -0.8787410855293274, -0.14260819554328918, 0.18247509002685547, -0.18025749921798706, -0.7098209857940674, 0.1126750111579895, -0.0332549624145031, 0.7724456787109375, 0.3981875479221344, -0.1899879276752472, -0.14864680171012878, -0.6625394821166992, 0.26694315671920776, -0.5815560817718506, 0.20814023911952972, 0.7924160361289978, 0.07765793800354004, 1.486771821975708, -0.15541541576385498, -0.4328983426094055, -1.0767279863357544, -0.2141694873571396, -0.10612881928682327, 0.140322744846344, -1.0572969913482666, -0.4731893241405487, -1.4977514743804932, -0.4744597375392914, -1.4130780696868896, -2.4262404441833496, 0.06300709396600723, 0.92876756191253

## Visualização dos Resultados

In [28]:
import pandas as pd

df = pd.DataFrame(resultados)[['id', 'score', 'texto']]
print("\nResultados em Tabela:")
print(df.to_string(index=False))
print(f"\nMelhor match score: {df['score'].max():.4f}")


Resultados em Tabela:
id    score                                                                texto
 1 0.032787    Lei de Acesso à Informação garante direito a informações públicas
 3 0.032258               LGPD protege dados pessoais e privacidade dos cidadãos
 2 0.015873 Direitos fundamentais incluem vida, liberdade, igualdade e segurança

Melhor match score: 0.0328


## Snippet Copiável para Reutilizar (BGE-M3 + RRF oficial)

In [29]:
snippet = '''
import os, requests
from opensearchpy import OpenSearch

OLLAMA_BASE_URL    = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "bge-m3")
EMBED_DIM = 1024

def ollama_embed(text):
    r = requests.post(f"{OLLAMA_BASE_URL}/api/embeddings",
                      json={"model": OLLAMA_EMBED_MODEL, "prompt": text}, timeout=30)
    return r.json()["embedding"]

client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "admin"),
    use_ssl=False, verify_certs=False,
)

# Pipeline RRF oficial OpenSearch 2.19+
rrf = {
    "description": "Post processor for hybrid RRF search",
    "phase_results_processors": [{
        "score-ranker-processor": {
            "combination": {"technique": "rrf", "rank_constant": 60}
        }
    }]
}
client.transport.perform_request("PUT", "/_search/pipeline/rrf_pipeline", body=rrf)

def hybrid_search(query, index, size=5):
    qe = ollama_embed(query)
    body = {
        "size": size,
        "query": {
            "hybrid": {
                "queries": [
                    {"match": {"texto": {"query": query}}},
                    {"knn": {"embedding": {"vector": qe, "k": size}}},
                ]
            }
        }
    }
    return client.search(index=index, body=body,
                         params={"search_pipeline": "rrf_pipeline"})
'''

print(snippet)


import os, requests
from opensearchpy import OpenSearch

OLLAMA_BASE_URL    = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "bge-m3")
EMBED_DIM = 1024

def ollama_embed(text):
    r = requests.post(f"{OLLAMA_BASE_URL}/api/embeddings",
                      json={"model": OLLAMA_EMBED_MODEL, "prompt": text}, timeout=30)
    return r.json()["embedding"]

client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "admin"),
    use_ssl=False, verify_certs=False,
)

# Pipeline RRF oficial OpenSearch 2.19+
rrf = {
    "description": "Post processor for hybrid RRF search",
    "phase_results_processors": [{
        "score-ranker-processor": {
            "combination": {"technique": "rrf", "rank_constant": 60}
        }
    }]
}
client.transport.perform_request("PUT", "/_search/pipeline/rrf_pipeline", body=rrf)

def hybrid_search(query, index, size=5):
    qe = ollama_embed(query)
    body = {
  

## Resumo

**Componentes**:
- ✓ Embedding BGE-M3 via Ollama (dim=1024)
- ✓ Índice híbrido com BM25 + KNN (HNSW/FAISS)
- ✓ Pipeline RRF com **`score-ranker-processor`** (sintaxe oficial OpenSearch 2.19+)
- ✓ Query combinada via `hybrid` + `?search_pipeline=...`
- ✓ Função reutilizável

**Próximos passos**:
- Integrar Ollama ao OpenSearch como **connector ML Commons** (ver LAB1) para embeddings server-side
- Adicionar Neural Sparse Search com modelo `opensearch-neural-sparse-encoding-multilingual-v1` (ver LAB4)
- Implementar Contextual Retrieval com Groq + Ollama (ver LAB5)